# imports

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import sys
import sklearn

sys.path.append('../src')
from load_clean_data import load_clean_data

In [2]:
df = load_clean_data()
print(df.shape)

c:\Users\corbi\Documents\Projects\sales-forecasting\sales-forecasting-project\notebooks\../src\load_clean_data.py:6: DtypeWarning: Columns (7: StateHoliday) have mixed types. Specify dtype option on import or set low_memory=False.
  train_data = pd.read_csv('../data/train.csv')


(1017209, 18)


# building the train-test split

train : dates up to june 30th 2015 

test: dates after june 30th 2015 up to july 30th 2015

In [6]:
store1 = df[df['Store'] == 1].sort_values('Date')

train  = store1[store1['Date'] < '2015-06-30']
test   = store1[store1['Date'] >= '2015-06-30']

In [7]:
length_check = len(train) + len(test) == len(store1)
order_check  = store1['Date'].is_monotonic_increasing

print(f"Length check: {length_check}, Order check: {order_check}")

Length check: True, Order check: True


In [13]:
train_series = train.set_index('Date')['Sales']
test_series  = test.set_index('Date')['Sales']

train_series = train_series.asfreq('D')
test_series = test_series.asfreq('D')

print(train_series.index)
print(test_series.index)
print(train_series.isnull().sum())
print(test_series.isnull().sum())

DatetimeIndex(['2013-01-01', '2013-01-02', '2013-01-03', '2013-01-04',
               '2013-01-05', '2013-01-06', '2013-01-07', '2013-01-08',
               '2013-01-09', '2013-01-10',
               ...
               '2015-06-20', '2015-06-21', '2015-06-22', '2015-06-23',
               '2015-06-24', '2015-06-25', '2015-06-26', '2015-06-27',
               '2015-06-28', '2015-06-29'],
              dtype='datetime64[us]', name='Date', length=910, freq='D')
DatetimeIndex(['2015-06-30', '2015-07-01', '2015-07-02', '2015-07-03',
               '2015-07-04', '2015-07-05', '2015-07-06', '2015-07-07',
               '2015-07-08', '2015-07-09', '2015-07-10', '2015-07-11',
               '2015-07-12', '2015-07-13', '2015-07-14', '2015-07-15',
               '2015-07-16', '2015-07-17', '2015-07-18', '2015-07-19',
               '2015-07-20', '2015-07-21', '2015-07-22', '2015-07-23',
               '2015-07-24', '2015-07-25', '2015-07-26', '2015-07-27',
               '2015-07-28', '2015-07-29

# SARIMA

### import

In [ ]:
from statsmodels.tsa.statespace.sarimax import SARIMAX

### setup

In [15]:
sarima_model = SARIMAX(train_series, 
                       order=(1, 1, 1), 
                       seasonal_order=(1, 1, 1, 7))

sarima_results = sarima_model.fit()

In [19]:
sarima_forecast = sarima_results.forecast(steps=len(test_series)) #type: ignore

sarima_forecast = sarima_forecast.clip(lower=0)

length_check = len(sarima_forecast) == len(test_series)
print(f"Length check: {length_check}")

Length check: True


In [20]:
print(sarima_forecast.head(10))
print(test_series.head(10))

2015-06-30    4622.776106
2015-07-01    4353.276278
2015-07-02    3917.249703
2015-07-03    4358.899833
2015-07-04    4818.950265
2015-07-05       0.000000
2015-07-06    4724.848108
2015-07-07    4484.255874
2015-07-08    4285.548274
2015-07-09    3893.117526
Freq: D, Name: predicted_mean, dtype: float64
Date
2015-06-30    5735
2015-07-01    5223
2015-07-02    5558
2015-07-03    4665
2015-07-04    4797
2015-07-05       0
2015-07-06    4359
2015-07-07    3650
2015-07-08    3797
2015-07-09    3897
Freq: D, Name: Sales, dtype: int64


### eval

In [21]:
RMSE = sklearn.metrics.root_mean_squared_error(test_series, sarima_forecast)
MAE = sklearn.metrics.mean_absolute_error(test_series, sarima_forecast)
WAPE = 100 * (abs(test_series - sarima_forecast).sum()) / (abs(test_series).sum())

print(f"RMSE: {RMSE:.2f}")
print(f"MAE: {MAE:.2f}")
print(f"WAPE: {WAPE:.2f}%")

RMSE: 703.38
MAE: 559.96
WAPE: 14.11%
